Install Required Libraries

In [1]:
!pip install -U langchain langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 1.3 MB/s eta 0:00:00


Import Libraries

In [2]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

Add Your Gemini API Key

In [3]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

Initialize the LLM

In [10]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

Create Strict Grounding Prompt

In [6]:
prompt = ChatPromptTemplate.from_template("""
You are a strict AI assistant.

Answer the question ONLY using the provided context.

If the answer cannot be found in the context,
reply exactly with:

"I do not have enough information"

Context:
{context}

Question:
{question}
""")

Create Context Data

In [5]:
context = """
The company allows employees to work from home up to 3 days per week.
Employees must attend office meetings on Mondays and Thursdays.
Work hours are from 9 AM to 5 PM.
"""

Create Cache Dictionary

In [7]:
query_cache = {}

Build the Smart Generation Function

In [8]:
def generate_answer(user_query):

    # Check Cache
    if user_query in query_cache:
        print("Returned from Cache:", query_cache[user_query])
        return query_cache[user_query]

    # Build prompt
    formatted_prompt = prompt.format(
        context=context,
        question=user_query
    )

    # Call LLM
    response = llm.invoke(formatted_prompt)

    answer = response.content

    # Store in cache
    query_cache[user_query] = answer

    return answer

Scenario 1 (First Ask)

In [11]:
question1 = "How many days can employees work from home?"

print("Scenario 1 (First Ask)")
print("Question:", question1)

response1 = generate_answer(question1)

print("LLM Response:", response1)

Scenario 1 (First Ask)
Question: How many days can employees work from home?
LLM Response: Employees can work from home up to 3 days per week.


Scenario 2 (Cache Hit)

In [12]:
print("\nScenario 2 (Cache Hit)")

response2 = generate_answer(question1)


Scenario 2 (Cache Hit)
Returned from Cache: Employees can work from home up to 3 days per week.


Scenario 3 (Grounding Test)

In [13]:
question3 = "What is the recipe for a chocolate cake?"

print("\nScenario 3 (Grounding Test)")
print("Question:", question3)

response3 = generate_answer(question3)

print("LLM Response:", response3)


Scenario 3 (Grounding Test)
Question: What is the recipe for a chocolate cake?
LLM Response: I do not have enough information
